## 1. Import Libraries

In [14]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from io import StringIO
from tqdm.notebook import tqdm
import time
import re
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=pd.errors.SettingWithCopyWarning)

print("Libraries imported successfully")

Libraries imported successfully


## 2. Load Bank Data

In [15]:
# Load data bank syariah dari CSV
df_banks = pd.read_csv('/Users/mraffyzeidan/Learning/xcap/ojk_cfs_bpr_syariah.csv')

print(f"Total banks loaded: {len(df_banks)}")
print(f"Provinsi available: {len(df_banks['Provinsi'].unique())} provinsi")
df_banks.head(10)

Total banks loaded: 194
Provinsi available: 25 provinsi


,Provinsi,Kabupaten/Kota,Nama Bank,Kode Bank
0,Provinsi Jawa Barat,Kab. Bekasi,PT Bank Perekonomian Rakyat Syariah Amanah Insani,620066
1,Provinsi Jawa Barat,Kab. Bogor,PT Bank Perekonomian Rakyat Syariah Amanah Ummah,620004
2,Provinsi Jawa Barat,Kab. Bogor,PT Bank Perekonomian Rakyat Syariah Botani Bin...,620061
3,Provinsi Jawa Barat,Kab. Bogor,PT. BPRS Rif'atul Ummah,620068
4,Provinsi Jawa Barat,Kab. Bogor,PT Bank Perekonomian Rakyat Syariah Harta Insa...,620069
5,Provinsi Jawa Barat,Kab. Bogor,PT BPRS Bogor Tegar Beriman,620177
6,Provinsi Jawa Barat,Kab. Cianjur,PT Bank Perekonomian Rakyat Syariah Gaido Indo...,620043
7,Provinsi Jawa Barat,Kab. Bandung,PT Bank Perekonomian Rakyat Syariah Amanah Rab...,620002
8,Provinsi Jawa Barat,Kab. Bandung,PT Bank Perekonomian Rakyat Syariah Almasoem,620027
9,Provinsi Jawa Barat,Kab. Bandung,PT Bank Perekonomian Rakyat Syariah Al Ihsan,620044


## 3. Configuration

In [16]:
BASE_URL = "https://cfs.ojk.go.id/cfs/ReportViewerForm.aspx"
MONTH = "12"  # Fixed: Desember
PERIOD_TYPE = "R"

# Report types untuk BPRS (2010-2018) - BERBEDA dari 2019-2024!
REPORT_TYPES_2010_2018 = {
    "BPS-900-000001": "Neraca",
    "BPS-900-000002": "Laba Rugi",
    "BPS-900-000003": "Komitmen dan Kontijensi",
    "BPS-900-000004": "KAP dan Informasi Lain",
    "BPS-900-000005": "Sumber dan Penggunaan ZIS",
    "BPS-900-000006": "Sumber dan Penggunaan Qardhul Hasan",
    "BPS-900-000007": "Distribusi Bagi Hasil",
    "BPS-900-000008": "Perubahan Dana Investasi Terikat"
}

# Column order based on Dataset Syakhsan.xlsx - BPRS (2010 - 2018)
# STRUKTUR SAMA dengan 2019-2024, hanya report type yang berbeda
COLUMN_ORDER = [
    'Tahun', 'Bulan', 'Longitude', 'Latitude', 'Nama_BPR', 
    'Kabupaten_Kota', 'Provinsi', 'Kode_Bank',
    # Neraca - Aset
    'Total_Aset',
    'Piutang_Murabahah',
    'Piutang_Istishna',
    'Piutang_Multijasa',
    'Piutang_Qardh',
    'Piutang_Sewa',
    'Pembiayaan_Mudharabah',
    'Pembiayaan_Musyarakah',
    'Pembiayaan_Lainnya',
    'Salam',
    'Agunan_yang_Diambil_Alih',
    # Neraca - Liabilitas
    'Tabungan_Wadiah',
    'Tabungan_Mudharabah',
    'Deposito_Mudharabah',
    'Cadangan_Kerugian_Penurunan_Nilai_1',
    'Cadangan_Kerugian_Penurunan_Nilai_2',
    'Liabilitas_Segera',
    'Liabilitas_kepada_BI',
    'Liabilitas_kepada_Bank_Lain',
    'Pembiayaan_Diterima',
    'Liabilitas_Lainnya',
    'Dana_Syirkah_Temporer',
    # Laba Rugi
    'Pendapatan_setelah_distribusi_bagi_hasil',
    'Pendapatan_dari_penyaluran_dana',
    'Beban_Operasional',
    'Zakat',
    'Laba_Rugi_Bersih',
    # Rasio Keuangan (dari KAP dan Informasi Lain)
    'KPMM',
    'NPF_Neto',
    'NPF_Gross',
    'ROA',
    'BOPO',
    'FDR',
    'Cash_Ratio',
    'NI'
]

print("✓ Configuration loaded")
print(f"Report types untuk 2010-2018: {len(REPORT_TYPES_2010_2018)}")

✓ Configuration loaded
Report types untuk 2010-2018: 8


## 4. Helper Functions

In [17]:
def clean_number(value):
    """Convert string number dengan format Indonesia ke float"""
    if pd.isna(value) or value == '' or value == 'NaN':
        return None
    
    if isinstance(value, (int, float)):
        return float(value)
    
    value = str(value).strip()
    value = re.sub(r'[^\d.,()?\-]', '', value)
    
    if '(' in value and ')' in value:
        value = '-' + value.replace('(', '').replace(')', '')
    
    value = value.replace(',', '')
    
    try:
        result = float(value)
        return result
    except:
        return None

def find_value_in_row(df, keywords, col_index=2, debug=False):
    """
    Cari nilai di tabel berdasarkan keyword
    Returns: float (including 0.0 if actual zero) or None (if not found)
    debug: Print debug info
    """
    for col in df.columns[:2]:
        for keyword in keywords:
            mask = df[col].astype(str).str.contains(keyword, case=False, regex=False, na=False)
            if mask.any():
                idx = df[mask].index[0]
                if col_index < len(df.columns):
                    value = clean_number(df.iloc[idx, col_index])
                    if debug:
                        if value == 0.0:
                            print(f"  ✓ Found '{keyword}' at row {idx}, value: {df.iloc[idx, col_index]} → 0 (actual zero)")
                        else:
                            print(f"  ✓ Found '{keyword}' at row {idx}, value: {value:,.0f}")
                    return value
                return None
    
    if debug:
        print(f"  ✗ Keywords {keywords} NOT FOUND")
    return None

print("Helper functions defined")

Helper functions defined


## 5. Extraction & Parsing Functions

In [18]:
def extract_table(bank_code_number, bank_code, year, report_type, table_index=16, timeout=15):
    """Ekstrak tabel dari laporan BPRS"""
    params = {
        'BankCodeNumber': bank_code_number,
        'BankCode': bank_code,
        'Month': MONTH,
        'Year': str(year),
        'FinancialReportPeriodTypeCode': PERIOD_TYPE,
        'FinancialReportTypeCode': report_type
    }
    
    try:
        response = requests.get(BASE_URL, params=params, timeout=timeout)
        response.raise_for_status()
        
        # Extract tables
        dfs = pd.read_html(StringIO(response.text))
        
        if not dfs or table_index >= len(dfs):
            return None
        
        return dfs[table_index]
        
    except Exception as e:
        return None

def parse_neraca(df, debug=False):
    """Parse Neraca BPRS (2010-2018)"""
    data = {}
    
    # Mapping fields untuk Neraca 2010-2018
    fields_mapping = {
        'Total_Aset': [['JUMLAH AKTIVA', 'Total Aset', 'JUMLAH ASET', 'Jumlah Aset'], 2],
        'Piutang_Murabahah': [['a. Piutang Murabahah', 'Piutang Murabahah', 'a.Piutang Murabahah'], 2],
        'Piutang_Istishna': [["Piutang Istishna'", 'b. Piutang Istishna', 'Piutang Istishna', 'b.Piutang Istishna'], 2],
        'Piutang_Multijasa': [['c. Piutang Multijasa', 'Piutang Multijasa', 'c.Piutang Multijasa'], 2],
        'Piutang_Qardh': [['Qardh', 'd. Piutang Qardh', 'Piutang Qardh', 'd.Piutang Qardh'], 2],
        'Piutang_Sewa': [['Ijarah', 'e. Piutang Sewa', 'Piutang Sewa', 'e.Piutang Sewa'], 2],
        'Pembiayaan_Mudharabah': [['a. Mudharabah', 'Mudharabah', 'a.Mudharabah'], 2],
        'Pembiayaan_Musyarakah': [['b. Musyarakah', 'Musyarakah', 'b.Musyarakah'], 2],
        'Pembiayaan_Lainnya': [['c. Lainnya', 'c.Lainnya'], 2],
        'Salam': [['Piutang Salam', '9. Salam', '8. Salam', 'Salam'], 2],
        'Agunan_yang_Diambil_Alih': [['12. Agunan yang Diambil Alih', 'Agunan yang Diambil Alih', '11. Agunan yang Diambil Alih'], 2],
        'Tabungan_Wadiah': [['2. Tabungan Wadiah', 'Tabungan Wadiah'], 2],
        'Tabungan_Mudharabah': [['a. Tabungan', 'Tabungan Mudharabah', 'a.Tabungan'], 2],
        'Deposito_Mudharabah': [['b. Deposito', 'Deposito Mudharabah', 'b.Deposito'], 2],
        'Liabilitas_Segera': [['Kewajiban Segera', '1. Kewajiban Segera', 'Liabilitas Segera'], 2],
        'Liabilitas_kepada_BI': [['Kewajiban Kepada Bank Indonesia', '4. Kewajiban Kepada Bank Indonesia', 'Liabilitas kepada Bank Indonesia'], 2],
        'Liabilitas_kepada_Bank_Lain': [['Kewajiban Lain-Lain', '5. Kewajiban Lain-Lain', 'Liabilitas kepada Bank Lain'], 2],
        'Pembiayaan_Diterima': [['Pembiayaan/Pinjaman Yang Diterima', '6. Pembiayaan Diterima', 'Pembiayaan Diterima'], 2],
        'Liabilitas_Lainnya': [['7. Liabilitas Lainnya', 'Liabilitas Lainnya'], 2],
        'Dana_Syirkah_Temporer': [['8. Dana Investasi Profit Sharing', 'Dana Syirkah Temporer', 'Dana Investasi Profit Sharing'], 2],
    }
    
    for field, (keywords, col_idx) in fields_mapping.items():
        if debug:
            print(f"Parsing {field}: {keywords}")
        value = find_value_in_row(df, keywords, col_idx, debug=debug)
        data[field] = value
    
    # Special handling untuk Cadangan Kerugian Penurunan Nilai
    # Di OJK: "Penyisihan Penghapusan Aktiva -/-" (total saja, tidak split)
    cadangan_total = find_value_in_row(df, ['Penyisihan Penghapusan Aktiva', 'Penyisihan Penghapusan Aset', 'PPAP'], 2, debug=debug)
    
    # Simpan ke kedua kolom dengan nilai yang sama (karena tidak split di 2010-2018)
    data['Cadangan_Kerugian_Penurunan_Nilai_1'] = cadangan_total
    data['Cadangan_Kerugian_Penurunan_Nilai_2'] = None  # Tidak ada split di periode ini
    
    # Calculate Cash_Ratio from 3 components (per Dataset Syakhsan.xlsx row 31)
    if debug:
        print(f"\nCalculating Cash_Ratio from 3 components:")
    
    kas = find_value_in_row(df, ['1. Kas', 'Kas'], 2, debug=debug)
    penempatan_bi = find_value_in_row(df, ['2. Penempatan Pada Bank Indonesia', 'Penempatan pada Bank Indonesia'], 2, debug=debug)
    penempatan_bank_lain = find_value_in_row(df, ['3. Penempatan Pada Bank Lain', 'Penempatan pada Bank Lain'], 2, debug=debug)
    
    # Sum all 3 components (treat None as 0 for calculation)
    cash_ratio_components = [kas or 0, penempatan_bi or 0, penempatan_bank_lain or 0]
    cash_ratio_sum = sum(cash_ratio_components)
    
    # Only set Cash_Ratio if at least one component was found (not all None)
    if kas is not None or penempatan_bi is not None or penempatan_bank_lain is not None:
        data['Cash_Ratio'] = cash_ratio_sum
        if debug:
            print(f"  ✓ Cash_Ratio calculated: {kas or 0:,.0f} + {penempatan_bi or 0:,.0f} + {penempatan_bank_lain or 0:,.0f} = {cash_ratio_sum:,.0f}")
    else:
        data['Cash_Ratio'] = None
        if debug:
            print(f"  ✗ Cash_Ratio: All components not found")
    
    return data

def parse_laba_rugi(df, debug=False):
    """Parse Laba Rugi BPRS (2010-2018)"""
    data = {}
    
    fields_mapping = {
        'Pendapatan_setelah_distribusi_bagi_hasil': [['III. PENDAPATAN OPERASIONAL SETELAH DISTRIBUSI BAGI HASIL', 'III. PENDAPATAN SETELAH DISTRIBUSI BAGI HASIL', 'III. Pendapatan setelah distribusi bagi hasil'], 2],
        'Pendapatan_dari_penyaluran_dana': [['I. PENDAPATAN OPERASIONAL', 'I. Pendapatan Operasional', 'Pendapatan Operasional'], 2],
        'Beban_Operasional': [['IV. BEBAN OPERASIONAL', 'V. BEBAN OPERASIONAL LAINNYA', 'V. Beban Operasional', 'Beban Operasional'], 2],
        'Zakat': [['IX. ZAKAT', 'X. ZAKAT', 'X. Zakat', 'Zakat'], 2],
        'Laba_Rugi_Bersih': [['XI. LABA (RUGI) TAHUN BERJALAN', 'XI. Laba Rugi Bersih', 'Laba Rugi Bersih', 'XII. LABA (RUGI) TAHUN BERJALAN'], 2],
    }
    
    for field, (keywords, col_idx) in fields_mapping.items():
        if debug:
            print(f"Parsing {field}: {keywords}")
        value = find_value_in_row(df, keywords, col_idx, debug=debug)
        data[field] = value
    
    return data

def parse_kap_informasi_lain(df, debug=False):
    """Parse KAP dan Informasi Lain untuk ratio keuangan (2010-2018)"""
    data = {}
    # Rasio keywords (based on Dataset Syakhsan.xlsx)
    ratio_mappings = {
        'KPMM': ['10. KPMM', 'KPMM', 'Kewajiban Penyediaan Modal Minimum'],
        'NPF_Neto': ['7.  Rasio Non Performing Financing', 'Rasio Non Performing Financing', 'NPF'],  # Only one NPF in 2010-2018
        'ROA': ['12. ROA', 'ROA', 'Return on Asset'],
        'FDR': ['11. FDR', 'FDR', 'Financing to Deposit Ratio'],
    }
    
    # BOPO, NPF_Gross, NI, Cash_Ratio tidak ada di periode 2010-2018
    
    for field_name, keywords in ratio_mappings.items():
        found = False
        for row_idx in range(len(df)):
            row_text = ' '.join([str(df.iloc[row_idx, col]) for col in range(len(df.columns))]).lower()
            
            for keyword in keywords:
                if keyword.lower() in row_text:
                    # Get last column value (ratio column)
                    for col_idx in range(len(df.columns) - 1, -1, -1):
                        val_str = str(df.iloc[row_idx, col_idx])
                        if val_str not in ['NaN', 'nan', ''] and any(c.isdigit() for c in val_str):
                            val = clean_number(val_str)
                            # Ratio values should be between -100 to 1000%
                            if val is not None and -100 < val < 1000:
                                if debug:
                                    if val == 0.0:
                                        print(f"  ✓ Found '{field_name}': {val_str} → 0 (actual zero)")
                                    else:
                                        print(f"  ✓ Found '{field_name}': {val}")
                                data[field_name] = val
                                found = True
                                break
                    if found:
                        break
            if found:
                break
        
        if not found:
            if debug:
                print(f"  ✗ '{field_name}' NOT FOUND")
            data[field_name] = None
    
    # Set variabel yang memang tidak ada di periode 2010-2018
    data['NPF_Gross'] = None  # Tidak ada split Neto/Gross di periode ini
    data['BOPO'] = None  # Tidak tersedia
    data['NI'] = None  # Tidak tersedia
    # Note: Cash_Ratio dihitung dari Neraca (di parse_neraca function)
    
    return data

print("✓ Extraction & Parsing functions defined")

✓ Extraction & Parsing functions defined


## 6. Main Scraper Class

In [19]:
class BPRS2010Scraper:
    """Scraper untuk BPR Syariah 2010-2018 dengan debug lengkap"""
    
    def __init__(self):
        self.results = []
        
    def scrape_single_bank(self, bank_row, year, debug=False):
        """Scrape data untuk satu bank di satu tahun"""
        bank_code_number = str(bank_row['Kode Bank'])
        bank_code = bank_row['Nama Bank']
        
        result = {
            'Tahun': int(year),
            'Bulan': MONTH,
            'Nama_BPR': bank_code,
            'Kabupaten_Kota': bank_row['Kabupaten/Kota'],
            'Provinsi': bank_row['Provinsi'],
            'Kode_Bank': bank_code_number,
            'Longitude': None,
            'Latitude': None,
            'status': 'failed'
        }
        
        try:
            if debug:
                print(f"\n{'='*70}")
                print(f"Bank: {bank_code}")
                print(f"Year: {year}")
                print(f"Code: {bank_code_number}")
                print(f"{'='*70}")
            
            # Neraca (BPS-900-000001) - table index 16
            if debug:
                print(f"\n📊 Extracting NERACA...")
            
            df_neraca = extract_table(bank_code_number, bank_code, year, "BPS-900-000001", table_index=16)
            
            if df_neraca is not None:
                neraca_data = parse_neraca(df_neraca, debug=debug)
                result.update(neraca_data)
                if debug:
                    print(f"  ✓ Neraca parsed: {len(neraca_data)} fields")
                    print(f"  Total Aset: {neraca_data.get('Total_Aset', 0):,.0f}")
            else:
                if debug:
                    print(f"  ✗ Neraca not found")
            
            time.sleep(0.1)
            
            # Laba Rugi (BPS-900-000002) - table index 16
            if debug:
                print(f"\n💰 Extracting LABA RUGI...")
            
            df_laba = extract_table(bank_code_number, bank_code, year, "BPS-900-000002", table_index=16)
            
            if df_laba is not None:
                laba_data = parse_laba_rugi(df_laba, debug=debug)
                result.update(laba_data)
                if debug:
                    print(f"  ✓ Laba Rugi parsed: {len(laba_data)} fields")
                    print(f"  Laba Rugi Bersih: {laba_data.get('Laba_Rugi_Bersih', 0):,.0f}")
            else:
                if debug:
                    print(f"  ✗ Laba Rugi not found")
            
            time.sleep(0.1)
            
            # KAP dan Informasi Lain (BPS-900-000004) - untuk ratio keuangan
            if debug:
                print(f"\n📈 Extracting KAP & INFORMASI LAIN (RASIO)...")
            
            df_kap = extract_table(bank_code_number, bank_code, year, "BPS-900-000004", table_index=16)
            
            if df_kap is not None:
                kap_data = parse_kap_informasi_lain(df_kap, debug=debug)
                result.update(kap_data)
                if debug:
                    print(f"  ✓ KAP & Info Lain parsed: {len(kap_data)} fields")
                    non_zero_ratios = [k for k, v in kap_data.items() if v != 0]
                    print(f"  Non-zero ratios: {non_zero_ratios}")
            else:
                if debug:
                    print(f"  ✗ KAP & Info Lain not found")
            
            result['status'] = 'success'
            
            if debug:
                print(f"\n✅ Status: SUCCESS")
            
        except Exception as e:
            result['error'] = str(e)
            if debug:
                print(f"  ✗ Error: {str(e)}")
        
        return result
    
    def run(self, df_banks_filtered, years, max_workers=4, debug_bank_idx=None, debug_year=None):
        """Run scraping dengan ThreadPoolExecutor"""
        tasks = []
        for bank_idx, (_, bank_row) in enumerate(df_banks_filtered.iterrows()):
            for year in years:
                # Debug mode: only debug specific bank & year
                is_debug = (debug_bank_idx is not None and bank_idx == debug_bank_idx and 
                           (debug_year is None or year == debug_year))
                tasks.append((bank_row, year, is_debug))
        
        total_tasks = len(tasks)
        results = []
        
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {executor.submit(self.scrape_single_bank, task[0], task[1], task[2]): task for task in tasks}
            
            with tqdm(total=total_tasks, desc="Scraping", unit="task") as pbar:
                for future in as_completed(futures):
                    try:
                        result = future.result()
                        results.append(result)
                    except Exception as e:
                        print(f"Task failed: {e}")
                    pbar.update(1)
        
        return results

print("BPRS2010Scraper class defined")

BPRS2010Scraper class defined


## 7. 🐛 DEBUG MODE - Single Bank Testing

**Test 1 bank untuk beberapa tahun sample untuk pastikan semua variabel terdeteksi:**

In [20]:
# ============================================================================
# DEBUG CONFIGURATION - TEST SATU BANK UNTUK SEMUA TAHUN
# ============================================================================

# Pilih bank pertama untuk testing
TEST_BANK_IDX = 0
TEST_YEARS = [2010, 2013, 2016, 2018]  # Sample tahun untuk testing

test_bank = df_banks.iloc[TEST_BANK_IDX]

print("="*70)
print(" DEBUG MODE - TESTING SINGLE BANK")
print("="*70)
print(f"Bank: {test_bank['Nama Bank']}")
print(f"Kode: {test_bank['Kode Bank']}")
print(f"Provinsi: {test_bank['Provinsi']}")
print(f"Testing Tahun: {TEST_YEARS}")
print("="*70)

 DEBUG MODE - TESTING SINGLE BANK
Bank: PT Bank Perekonomian Rakyat Syariah Amanah Insani
Kode: 620066
Provinsi: Provinsi Jawa Barat
Testing Tahun: [2010, 2013, 2016, 2018]


In [21]:
# Run debug scraping
scraper = BPRS2010Scraper()

debug_results = []
for year in TEST_YEARS:
    print(f"\n" + "#"*70)
    print(f"# Testing Year: {year}")
    print("#"*70)
    result = scraper.scrape_single_bank(test_bank, year, debug=True)
    debug_results.append(result)
    print(f"\nStatus: {result['status']}")


######################################################################
# Testing Year: 2010
######################################################################

Bank: PT Bank Perekonomian Rakyat Syariah Amanah Insani
Year: 2010
Code: 620066

📊 Extracting NERACA...
  ✗ Neraca not found

💰 Extracting LABA RUGI...
  ✗ Laba Rugi not found

📈 Extracting KAP & INFORMASI LAIN (RASIO)...
  ✗ KAP & Info Lain not found

✅ Status: SUCCESS

Status: success

######################################################################
# Testing Year: 2013
######################################################################

Bank: PT Bank Perekonomian Rakyat Syariah Amanah Insani
Year: 2013
Code: 620066

📊 Extracting NERACA...
Parsing Total_Aset: ['JUMLAH AKTIVA', 'Total Aset', 'JUMLAH ASET', 'Jumlah Aset']
  ✓ Found 'JUMLAH AKTIVA' at row 21, value: 52,305,114
Parsing Piutang_Murabahah: ['a. Piutang Murabahah', 'Piutang Murabahah', 'a.Piutang Murabahah']
  ✓ Found 'Piutang Murabahah' at row 7, value

In [22]:
# Show debug results summary
print("\n" + "="*70)
print(" DEBUG RESULTS SUMMARY")
print("="*70)

df_debug = pd.DataFrame(debug_results)

# Pisah variabel per kategori
neraca_cols = ['Total_Aset', 'Piutang_Murabahah', 'Piutang_Istishna', 'Piutang_Multijasa', 
               'Piutang_Qardh', 'Piutang_Sewa', 'Pembiayaan_Mudharabah', 'Pembiayaan_Musyarakah',
               'Pembiayaan_Lainnya', 'Salam', 'Agunan_yang_Diambil_Alih',
               'Tabungan_Wadiah', 'Tabungan_Mudharabah', 'Deposito_Mudharabah',
               'Cadangan_Kerugian_Penurunan_Nilai_1', 'Cadangan_Kerugian_Penurunan_Nilai_2',
               'Liabilitas_Segera', 'Liabilitas_kepada_BI', 'Liabilitas_kepada_Bank_Lain',
               'Pembiayaan_Diterima', 'Liabilitas_Lainnya', 'Dana_Syirkah_Temporer']

laba_rugi_cols = ['Pendapatan_setelah_distribusi_bagi_hasil', 'Pendapatan_dari_penyaluran_dana',
                  'Beban_Operasional', 'Zakat', 'Laba_Rugi_Bersih']

ratio_cols = ['KPMM', 'NPF_Neto', 'NPF_Gross', 'ROA', 'BOPO', 'FDR', 'Cash_Ratio', 'NI']

print("\n✓ NERACA VARIABLES:")
print("="*70)
for col in neraca_cols:
    if col in df_debug.columns:
        non_zero = (df_debug[col] != 0).sum()
        total = len(df_debug)
        pct = (non_zero / total * 100) if total > 0 else 0
        status = "✓" if non_zero > 0 else "✗"
        print(f"  {status} {col:45s}: {non_zero}/{total} ({pct:.0f}%)")

print("\n✓ LABA RUGI VARIABLES:")
print("="*70)
for col in laba_rugi_cols:
    if col in df_debug.columns:
        non_zero = (df_debug[col] != 0).sum()
        total = len(df_debug)
        pct = (non_zero / total * 100) if total > 0 else 0
        status = "✓" if non_zero > 0 else "✗"
        print(f"  {status} {col:45s}: {non_zero}/{total} ({pct:.0f}%)")

print("\n✓ RATIO VARIABLES:")
print("="*70)
for col in ratio_cols:
    if col in df_debug.columns:
        non_zero = (df_debug[col] != 0).sum()
        total = len(df_debug)
        pct = (non_zero / total * 100) if total > 0 else 0
        status = "✓" if non_zero > 0 else "✗"
        print(f"  {status} {col:45s}: {non_zero}/{total} ({pct:.0f}%)")

print("\n" + "="*70)
print(" SAMPLE DATA")
print("="*70)
display(df_debug[['Tahun', 'Nama_BPR', 'Total_Aset', 'Laba_Rugi_Bersih', 'ROA', 'NPF_Gross']])


 DEBUG RESULTS SUMMARY

✓ NERACA VARIABLES:
  ✓ Total_Aset                                   : 4/4 (100%)
  ✓ Piutang_Murabahah                            : 4/4 (100%)
  ✓ Piutang_Istishna                             : 1/4 (25%)
  ✓ Piutang_Multijasa                            : 1/4 (25%)
  ✓ Piutang_Qardh                                : 4/4 (100%)
  ✓ Piutang_Sewa                                 : 1/4 (25%)
  ✓ Pembiayaan_Mudharabah                        : 4/4 (100%)
  ✓ Pembiayaan_Musyarakah                        : 1/4 (25%)
  ✓ Pembiayaan_Lainnya                           : 4/4 (100%)
  ✓ Salam                                        : 1/4 (25%)
  ✓ Agunan_yang_Diambil_Alih                     : 4/4 (100%)
  ✓ Tabungan_Wadiah                              : 4/4 (100%)
  ✓ Tabungan_Mudharabah                          : 1/4 (25%)
  ✓ Deposito_Mudharabah                          : 4/4 (100%)
  ✓ Cadangan_Kerugian_Penurunan_Nilai_1          : 4/4 (100%)
  ✓ Cadangan_Kerugian_Penurunan

,Tahun,Nama_BPR,Total_Aset,Laba_Rugi_Bersih,ROA,NPF_Gross
0,2010,PT Bank Perekonomian Rakyat Syariah Amanah Insani,NaN,NaN,NaN,NaN
1,2013,PT Bank Perekonomian Rakyat Syariah Amanah Insani,52305114.0,877804.0,2.00,NaN
2,2016,PT Bank Perekonomian Rakyat Syariah Amanah Insani,79614196.0,870078.0,1.09,NaN
3,2018,PT Bank Perekonomian Rakyat Syariah Amanah Insani,83748463.0,-2178386.0,-2.31,NaN


## 8. USER CONFIGURATION

**Edit parameter di bawah ini untuk full scraping:**

In [23]:
# ============================================================================
# EDIT PARAMETER DI SINI
# ============================================================================

# Rentang tahun (2010-2018)
YEAR_START = 2010
YEAR_END = 2018

# Pilih provinsi:
# - ['SEMUA'] untuk scrape semua provinsi
# - Atau list provinsi spesifik: ['Provinsi Jawa Barat', 'Provinsi DKI Jakarta']
SELECTED_PROVINCES = ['Provinsi Jawa Barat']

# Jumlah thread workers (4-8 optimal)
MAX_WORKERS = 8

# Nama file output
OUTPUT_FILENAME = 'bprs_2010_2018_financial_data'

# ============================================================================

print("Configuration set:")
print(f"   Tahun: {YEAR_START} - {YEAR_END}")
print(f"   Provinsi: {SELECTED_PROVINCES}")
print(f"   Workers: {MAX_WORKERS}")

Configuration set:
   Tahun: 2010 - 2018
   Provinsi: ['Provinsi Jawa Barat']
   Workers: 8


## 9. RUN FULL SCRAPING

**Jalankan cell ini untuk memulai scraping full:**

In [24]:
# Prepare data
years = list(range(YEAR_START, YEAR_END + 1))

# Filter banks by province
if 'SEMUA' in SELECTED_PROVINCES:
    filtered_banks = df_banks.copy()
else:
    filtered_banks = df_banks[df_banks['Provinsi'].isin(SELECTED_PROVINCES)].copy()

total_tasks = len(filtered_banks) * len(years)

print("="*70)
print(" STARTING FULL SCRAPING - BPR SYARIAH 2010-2018")
print("="*70)
print(f"Tahun: {YEAR_START} - {YEAR_END} ({len(years)} tahun)")
print(f"Bulan: Desember (Fixed)")
print(f"Provinsi: {', '.join(SELECTED_PROVINCES)}")
print(f"Jumlah Bank: {len(filtered_banks)}")
print(f"Total Tasks: {total_tasks}")
print(f"Workers: {MAX_WORKERS}")
print("="*70)

if total_tasks == 0:
    print("\n⚠️  WARNING: No banks found for selected provinces!")
    print(f"Available provinces in BPR Syariah data:")
    for prov in sorted(df_banks['Provinsi'].unique()):
        count = len(df_banks[df_banks['Provinsi'] == prov])
        print(f"  - {prov}: {count} banks")
else:
    print()
    
    # Run scraping
    start_time = time.time()
    scraper_full = BPRS2010Scraper()
    results = scraper_full.run(filtered_banks, years, max_workers=MAX_WORKERS)
    elapsed_time = time.time() - start_time
    
    # Process results
    df_results = pd.DataFrame(results)
    
    # Add missing columns with None (not 0!)
    for col in COLUMN_ORDER:
        if col not in df_results.columns:
            df_results[col] = None
    
    # Reorder columns
    available_cols = [col for col in COLUMN_ORDER if col in df_results.columns]
    df_results = df_results[available_cols]
    
    # Statistics
    success_count = (df_results['status'] == 'success').sum() if 'status' in df_results.columns else len(df_results)
    failed_count = total_tasks - success_count
    
    print("\n" + "="*70)
    print(" SCRAPING COMPLETED")
    print("="*70)
    print(f"Total Tasks: {total_tasks}")
    print(f"Success: {success_count} ({success_count/total_tasks*100:.1f}%)")
    print(f"Failed: {failed_count} ({failed_count/total_tasks*100:.1f}%)")
    print(f"Elapsed Time: {elapsed_time:.2f} seconds")
    print(f"Average: {elapsed_time/total_tasks:.2f} sec/task")
    print("="*70)
    
    # Remove status column
    df_export = df_results.drop(columns=['status'], errors='ignore')
    
    # DON'T replace None/0 with empty string - keep actual values!
    # Pandas will show None as NaN, but that's correct for missing data
    
    # Save files
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    csv_file = f"{OUTPUT_FILENAME}_{timestamp}.csv"
    df_export.to_csv(csv_file, index=False, encoding='utf-8-sig')
    print(f"\n✓ Saved: {csv_file}")
    
    try:
        excel_file = f"{OUTPUT_FILENAME}_{timestamp}.xlsx"
        df_export.to_excel(excel_file, index=False, engine='openpyxl')
        print(f"✓ Saved: {excel_file}")
    except Exception as e:
        print(f"Excel save failed: {e}")
    
    print("\n✓ Done!")

 STARTING FULL SCRAPING - BPR SYARIAH 2010-2018
Tahun: 2010 - 2018 (9 tahun)
Bulan: Desember (Fixed)
Provinsi: Provinsi Jawa Barat
Jumlah Bank: 34
Total Tasks: 306
Workers: 8



Scraping:   0%|          | 0/306 [00:00<?, ?task/s]


 SCRAPING COMPLETED
Total Tasks: 306
Success: 306 (100.0%)
Failed: 0 (0.0%)
Elapsed Time: 753.96 seconds
Average: 2.46 sec/task

✓ Saved: bprs_2010_2018_financial_data_20260105_113334.csv
✓ Saved: bprs_2010_2018_financial_data_20260105_113334.xlsx

✓ Done!


## 10. Data Quality Check

**Verifikasi coverage setiap variabel per kategori:**

In [25]:
df_export.describe().T

,count,mean,std,min,25%,50%,75%,max
Tahun,306.0,2.014000e+03,2.586218e+00,2010.00,2.012000e+03,2.014000e+03,2.016000e+03,2.018000e+03
Total_Aset,220.0,8.200668e+07,1.434549e+08,3434209.00,1.225078e+07,3.302976e+07,8.287929e+07,1.218331e+09
Piutang_Murabahah,220.0,5.130268e+07,1.075392e+08,286287.00,6.805915e+06,1.635262e+07,4.831820e+07,9.230535e+08
Piutang_Istishna,220.0,1.688587e+05,9.743648e+05,0.00,0.000000e+00,0.000000e+00,0.000000e+00,1.050041e+07
Piutang_Multijasa,220.0,6.992993e+06,1.972544e+07,0.00,0.000000e+00,2.304855e+05,2.134862e+06,1.301920e+08
Piutang_Qardh,220.0,1.001168e+06,2.777958e+06,0.00,0.000000e+00,1.347285e+05,7.366028e+05,2.150977e+07
Piutang_Sewa,220.0,6.481573e+04,2.629640e+05,0.00,0.000000e+00,0.000000e+00,0.000000e+00,1.991965e+06
Pembiayaan_Mudharabah,220.0,9.378296e+05,3.820410e+06,0.00,0.000000e+00,0.000000e+00,3.178020e+05,3.524542e+07
Pembiayaan_Musyarakah,220.0,2.907670e+06,6.661037e+06,0.00,0.000000e+00,2.505000e+05,2.367667e+06,4.032113e+07
Pembiayaan_Lainnya,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Variabel dari Neraca yang critical
neraca_cols = ['Total_Aset', 'Piutang_Murabahah', 'Piutang_Istishna', 'Piutang_Multijasa',
               'Piutang_Qardh', 'Piutang_Sewa', 'Pembiayaan_Mudharabah', 'Pembiayaan_Musyarakah',
               'Pembiayaan_Lainnya', 'Salam', 'Agunan_yang_Diambil_Alih',
               'Tabungan_Wadiah', 'Tabungan_Mudharabah', 'Deposito_Mudharabah',
               'Cadangan_Kerugian_Penurunan_Nilai_1', 'Cadangan_Kerugian_Penurunan_Nilai_2',
               'Liabilitas_Segera', 'Liabilitas_kepada_BI', 'Liabilitas_kepada_Bank_Lain',
               'Pembiayaan_Diterima', 'Liabilitas_Lainnya', 'Dana_Syirkah_Temporer']

# Variabel dari Laba Rugi
laba_rugi_cols = ['Pendapatan_setelah_distribusi_bagi_hasil', 'Pendapatan_dari_penyaluran_dana',
                  'Beban_Operasional', 'Zakat', 'Laba_Rugi_Bersih']

# Variabel Rasio
ratio_cols = ['KPMM', 'NPF_Neto', 'NPF_Gross', 'ROA', 'BOPO', 'FDR', 'Cash_Ratio', 'NI']

print("✓ NERACA VARIABLES:")
print("="*70)
for col in neraca_cols:
    if col in df_export.columns:
        # Count rows where value is not None/NaN
        non_null = df_export[col].notna().sum()
        total = len(df_export)
        pct = (non_null / total * 100) if total > 0 else 0
        print(f"{col:45s}: {non_null:4d}/{total:4d} ({pct:5.1f}%)")

print("\n✓ LABA RUGI VARIABLES:")
print("="*70)
for col in laba_rugi_cols:
    if col in df_export.columns:
        non_null = df_export[col].notna().sum()
        total = len(df_export)
        pct = (non_null / total * 100) if total > 0 else 0
        print(f"{col:45s}: {non_null:4d}/{total:4d} ({pct:5.1f}%)")

print("\n✓ RATIO VARIABLES:")
print("="*70)
for col in ratio_cols:
    if col in df_export.columns:
        non_null = df_export[col].notna().sum()
        total = len(df_export)
        pct = (non_null / total * 100) if total > 0 else 0
        print(f"{col:45s}: {non_null:4d}/{total:4d} ({pct:5.1f}%)")

✓ NERACA VARIABLES:
Total_Aset                                   : 1746/1746 (100.0%)
Piutang_Murabahah                            : 1745/1746 ( 99.9%)
Piutang_Istishna                             :  659/1746 ( 37.7%)
Piutang_Multijasa                            : 1312/1746 ( 75.1%)
Piutang_Qardh                                : 1235/1746 ( 70.7%)
Piutang_Sewa                                 :  808/1746 ( 46.3%)
Pembiayaan_Mudharabah                        : 1172/1746 ( 67.1%)
Pembiayaan_Musyarakah                        : 1247/1746 ( 71.4%)
Pembiayaan_Lainnya                           : 1746/1746 (100.0%)
Salam                                        :  618/1746 ( 35.4%)
Agunan_yang_Diambil_Alih                     : 1746/1746 (100.0%)
Tabungan_Wadiah                              : 1630/1746 ( 93.4%)
Tabungan_Mudharabah                          : 1602/1746 ( 91.8%)
Deposito_Mudharabah                          : 1732/1746 ( 99.2%)
Cadangan_Kerugian_Penurunan_Nilai_1          : 1746/1746

## 11. Summary by Year

**Check data availability per tahun:**

In [ ]:
print("\n✓ SUMMARY BY YEAR:")
print("="*70)
if 'Tahun' in df_export.columns:
    for year in sorted(df_export['Tahun'].unique()):
        df_year = df_export[df_export['Tahun'] == year]
        # Count where Total_Aset is not None
        success = df_year['Total_Aset'].notna().sum()
        total = len(df_year)
        pct = (success / total * 100) if total > 0 else 0
        print(f"Tahun {year}: {success:3d}/{total:3d} banks ({pct:5.1f}%) have data")
        
        # Detail untuk ratio
        ratio_available = df_year['ROA'].notna().sum()
        print(f"          Ratio data: {ratio_available:3d}/{total:3d} banks ({ratio_available/total*100:5.1f}%)")
        print()


✓ SUMMARY BY YEAR:
Tahun 2010: 194/194 banks (100.0%) have data
          Ratio data: 194/194 banks (100.0%)

Tahun 2011: 194/194 banks (100.0%) have data
          Ratio data: 181/194 banks ( 93.3%)

Tahun 2012: 194/194 banks (100.0%) have data
          Ratio data: 183/194 banks ( 94.3%)

Tahun 2013: 194/194 banks (100.0%) have data
          Ratio data: 192/194 banks ( 99.0%)

Tahun 2014: 194/194 banks (100.0%) have data
          Ratio data: 189/194 banks ( 97.4%)

Tahun 2015: 194/194 banks (100.0%) have data
          Ratio data: 190/194 banks ( 97.9%)

Tahun 2016: 194/194 banks (100.0%) have data
          Ratio data: 191/194 banks ( 98.5%)

Tahun 2017: 194/194 banks (100.0%) have data
          Ratio data: 190/194 banks ( 97.9%)

Tahun 2018: 194/194 banks (100.0%) have data
          Ratio data: 191/194 banks ( 98.5%)



## 12. Preview Results

In [ ]:
print("Sample Data (First 10 rows):")
df_export[['Tahun', 'Nama_BPR', 'Provinsi', 'Total_Aset', 'Laba_Rugi_Bersih', 'ROA', 'NPF_Gross']].head(10)

In [ ]:
# Sort dan save final version
df_export = df_export.sort_values(by=['Tahun', 'Provinsi', 'Kabupaten_Kota', 'Nama_BPR']).reset_index(drop=True)
df_export.to_csv('Full1018BPRS.csv', index=False)
print("✓ Saved sorted data: Full1018BPRS.csv")d

In [38]:
df_export.describe().T

,count,mean,std,min,25%,50%,75%,max
Tahun,1746.0,2.014000e+03,2.582729e+00,2010.0,2012.00,2014.0,2016.00,2.018000e+03
Total_Aset,1132.0,4.853544e+07,9.360128e+07,335764.0,9999589.00,21359541.0,46472830.25,1.218331e+09
Piutang_Murabahah,1132.0,2.761626e+07,6.086754e+07,0.0,5482179.75,11469124.5,25133774.75,9.230535e+08
Piutang_Istishna,1132.0,1.041372e+05,1.093527e+06,0.0,0.00,0.0,0.00,2.093975e+07
Piutang_Multijasa,1132.0,2.834178e+06,1.052380e+07,0.0,0.00,139511.0,1049634.75,1.301920e+08
Piutang_Qardh,1132.0,7.227670e+05,4.024882e+06,0.0,0.00,5774.5,205509.00,9.493716e+07
Piutang_Sewa,1132.0,9.201427e+04,8.057740e+05,0.0,0.00,0.0,0.00,2.089710e+07
Pembiayaan_Mudharabah,1132.0,8.092998e+05,2.505612e+06,0.0,0.00,0.0,481375.00,3.524542e+07
Pembiayaan_Musyarakah,1132.0,3.673614e+06,1.474709e+07,0.0,0.00,83700.0,1858437.50,2.038902e+08
Pembiayaan_Lainnya,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


---

## 📝 Notes:

### ⚠️ Perbedaan dengan periode 2019-2024:
- **Report Type Code**: `BPS-900-000XXX` (bukan BPS-901-XXX)
- **Report Names**: 
  - Neraca (bukan "Laporan Posisi Keuangan Publikasi")
  - Laba Rugi (bukan "Laporan Laba Rugi Publikasi")
  - KAP dan Informasi Lain (untuk ratio, bukan "Rasio Keuangan")

### Key Features:
- ✅ **Debug Mode** - Test 1 bank dulu sebelum scrape semua
- ✅ **Variable Detection** - Debug output untuk setiap variabel
- ✅ **Quality Check** - Verifikasi coverage per kategori variabel
- ✅ **Thread-safe** - Multi-threading untuk speed
- ✅ **Year Summary** - Breakdown data availability per tahun

### Tips:
1. **Selalu jalankan Debug Mode dulu** (Section 7) sebelum full scraping
2. Cek apakah semua variabel critical terdeteksi
3. Start dengan `MAX_WORKERS=4` untuk stability
4. Untuk dataset besar, scrape per provinsi

### Troubleshooting:
- **Variabel tidak terdeteksi?** → Cek keywords di parse functions, mungkin format berbeda per tahun
- **Banyak failed?** → Kurangi MAX_WORKERS atau cek internet
- **Ratio masih 0?** → Normal untuk tahun-tahun lama, tidak semua bank punya ratio data

---

In [22]:
# Quick check - print last debug result details
if debug_results:
    last_result = debug_results[-1]
    print("Last Year Result:")
    print(f"  Year: {last_result['Tahun']}")
    print(f"  Total_Aset: {last_result.get('Total_Aset', 'MISSING')}")
    print(f"  Laba_Rugi_Bersih: {last_result.get('Laba_Rugi_Bersih', 'MISSING')}")
    print(f"  ROA: {last_result.get('ROA', 'MISSING')}")
    print(f"  Status: {last_result['status']}")
    
    # Check if values are actual zeros or None
    print("\nActual types:")
    print(f"  Total_Aset type: {type(last_result.get('Total_Aset'))}, value: {repr(last_result.get('Total_Aset'))}")
    print(f"  Laba_Rugi_Bersih type: {type(last_result.get('Laba_Rugi_Bersih'))}, value: {repr(last_result.get('Laba_Rugi_Bersih'))}")

Last Year Result:
  Year: 2018
  Total_Aset: 83748463.0
  Laba_Rugi_Bersih: -2178386.0
  ROA: -2.31
  Status: success

Actual types:
  Total_Aset type: <class 'float'>, value: 83748463.0
  Laba_Rugi_Bersih type: <class 'float'>, value: -2178386.0
